# Binary Classification with CNNs and Pretraining: Chest X-Ray Images (Pneumonia)

## Setup

In [ ]:
# One-time setup: make the `visualization` helper importable, then fetch data +
# resolve paths. Each study's fetch logic lives in its own download_data.py.
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("ConvolutedComputerVision"):
        !git clone -q https://github.com/samlowe106/ConvolutedComputerVision.git
    sys.path.insert(0, "ConvolutedComputerVision/src")

from visualization import colab_bootstrap

DATA_ROOT, CKPT_ROOT = colab_bootstrap(study="chest-x-ray-images-pneumonia")

In [1]:
import numpy as np
import tensorflow as tf


import datetime

notebook_start_time = datetime.datetime.now()

I0000 00:00:1782627612.930092   10803 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782627614.267167   10803 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
test_dir = f"{DATA_ROOT}/test"
train_dir = f"{DATA_ROOT}/train"
validation_dir = f"{DATA_ROOT}/val"

batch_size = 32

# this api defaults to rgb images, even though they're actually grayscale
height, width, channels = 150, 150, 3

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    shuffle=False,
)

# because the outputs are batched, we have to concatenate all the batches
y_true = np.concatenate([y for x, y in test_ds], axis=0)

In [3]:
from visualization import summary_graphics


def get_class_training_weights(train_ds, normalize=True):
    labels, counts = np.unique(
        np.concatenate([y for x, y in train_ds], axis=0), return_counts=True
    )
    total = sum(counts)

    weights = [total / (2 * count) for count in counts]
    max_weight = np.max(weights)

    if normalize:
        return {label: weights[i] / max_weight for i, label in enumerate(labels)}

    return {label: weights[i] for i, label in enumerate(labels)}

In [4]:
class_weight = get_class_training_weights(train_ds=train_ds)

print(f"Weight for normal class: {class_weight[0]:1.3f}")
print(f"Weight for pneumonia class: {class_weight[1]:1.3f}")

Weight for normal class: 1.000
Weight for pneumonia class: 0.348


In [5]:
vgg19 = tf.keras.applications.VGG19()  # default parameters are fine
vgg19.summary()

Model: "vgg19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,31

 Total params: 143,667,240 (548.05 MB)

 Trainable params: 143,667,240 (548.05 MB)

 Non-trainable params: 0 (0.00 B)

## Training

In [6]:
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.InputLayer((height, width, channels), name="input"),
        # tf.keras.layers.RandomFlip("horizontal", name="0.1rflip"),
        tf.keras.layers.RandomRotation(0.2, name="0.2rrot"),
        tf.keras.layers.RandomBrightness(0.2, name="0.3rbright"),
        tf.keras.layers.RandomContrast(0.2, name="0.4rcont"),
        tf.keras.layers.RandomTranslation(0.2, 0.2, name="0.5rtran"),
        tf.keras.layers.RandomZoom(0.2, name="0.6rzoom"),
        # the model "starts" after the augmentation layers
        tf.keras.layers.Rescaling(1.0 / 255, name="rescale"),
        # vgg19 expects 244x244 images
        tf.keras.layers.Resizing(244, 244, name="resize"),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.1conv"
        ),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="1.3pool"),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.1conv"
        ),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="2.3pool"),
        # the xception paper says every convolutional layer is followed by a batch normalization layer
        # i'm willing to bet modern techniques might have moved away from this, but we'll try it anyway
        # however, we make an exception for the pretrained layers, since they're already trained
        # the rest of the convolutional layers in Xception are blocks of 2-3 depthwise separable layers, followed by a pooling layer
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.5conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.6batchnorm"),
        tf.keras.layers.MaxPooling2D((2, 2), name="3.7pool"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.5conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.6batchnorm"),
        tf.keras.layers.MaxPooling2D((2, 2), name="4.7pool"),
        # all the CNN models end with dense layers, probably to help the model "interpret" what the convolutional layers see
        tf.keras.layers.Flatten(name="5.1flatten"),
        # we almost certainly don't need these many neurons, but nain used that many and maybe he knows something I don't
        tf.keras.layers.Dense(1024, activation="relu", name="5.2dense"),
        # the above dense layer has 118 million of the 119 million trainable parameters!!!
        # add in a dropout layer with a high rate to prevent overfitting
        tf.keras.layers.Dropout(0.7, name="5.3dropout"),
        tf.keras.layers.Dense(512, activation="relu", name="5.4dense"),
        tf.keras.layers.Dropout(0.7, name="5.5dropout"),
        # 1 output neuron with sigmoid is equivalent to 2 output neurons with softmax, but trains faster:
        # https://stats.stackexchange.com/questions/207049/neural-network-for-binary-classification-use-1-or-2-output-neurons
        # tf.keras.layers.Dense(2, activation="softmax", name="output")
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ],
    name="Model_1",
)

# let's try AdamW
model.compile(
    optimizer=tf.keras.optimizers.AdamW(),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.TruePositives(name="tp"),
        tf.keras.metrics.TrueNegatives(name="tn"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    ],
)

model.get_layer("1.1conv").set_weights(vgg19.get_layer("block1_conv1").get_weights())
model.get_layer("1.2conv").set_weights(vgg19.get_layer("block1_conv2").get_weights())
model.get_layer("2.1conv").set_weights(vgg19.get_layer("block2_conv1").get_weights())
model.get_layer("2.2conv").set_weights(vgg19.get_layer("block2_conv2").get_weights())
model.get_layer("1.1conv").trainable = False
model.get_layer("1.2conv").trainable = False
model.get_layer("2.1conv").trainable = False
model.get_layer("2.2conv").trainable = False

model.summary()

Model: "Model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 0.2rrot (RandomRotation)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 0.3rbright (RandomBrightness)   │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 0.4rcont (RandomContrast)       │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 0.5rtran (RandomTranslation)    │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 0.6rzoom (RandomZoom)           │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescale (Rescaling)             │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resize (Resizing)               │ (None, 244, 244, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 1.1conv (Conv2D)                │ (None, 244, 244, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 1.2conv (Conv2D)                │ (None, 244, 244, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 1.3pool (MaxPooling2D)          │ (None, 122, 122, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 2.1conv (Conv2D)                │ (None, 122, 122, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 2.2conv (Conv2D)                │ (None, 122, 122, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 2.3pool (MaxPooling2D)          │ (None, 61, 61, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.1conv (SeparableConv2D)       │ (None, 61, 61, 256)    │        34,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.2batchnorm                    │ (None, 61, 61, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.3conv (SeparableConv2D)       │ (None, 61, 61, 256)    │        68,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.4batchnorm                    │ (None, 61, 61, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.5conv (SeparableConv2D)       │ (None, 61, 61, 256)    │        68,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.6batchnorm                    │ (None, 61, 61, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 3.7pool (MaxPooling2D)          │ (None, 30, 30, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 4.1conv (SeparableConv2D)       │ (None, 30, 30, 512)    │       133,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 4.2batchnorm                    │ (None, 30, 30, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 4.3conv (SeparableConv2D)       │ (None, 30, 30, 512)    │       267,26

 Total params: 119,599,297 (456.24 MB)

 Trainable params: 119,334,529 (455.23 MB)

 Non-trainable params: 264,768 (1.01 MB)

In [ ]:
def make_callbacks(filepath):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=filepath, save_best_only=True, monitor="val_loss", mode="min"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.5, patience=3
        ),
    ]


history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks(os.path.join(CKPT_ROOT, "best_model_transfer_1.keras")),
)

In [ ]:
summary_graphics(history, model, test_ds)

In [ ]:
# maybe the vgg16 weights are better
vgg16 = tf.keras.applications.VGG16()  # default parameters are fine
vgg16.summary()

In [ ]:
# remove the last batch normalization layers from each block, like Nain did
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.InputLayer((height, width, channels), name="input"),
        # tf.keras.layers.RandomFlip("horizontal", name="0.1rflip"),
        tf.keras.layers.RandomRotation(0.2, name="0.2rrot"),
        tf.keras.layers.RandomBrightness(0.2, name="0.3rbright"),
        tf.keras.layers.RandomContrast(0.2, name="0.4rcont"),
        tf.keras.layers.RandomTranslation(0.2, 0.2, name="0.5rtran"),
        tf.keras.layers.RandomZoom(0.2, name="0.6rzoom"),
        # the model "starts" after the augmentation layers
        tf.keras.layers.Rescaling(1.0 / 255, name="rescale"),
        # vgg19 expects 244x244 images
        tf.keras.layers.Resizing(244, 244, name="resize"),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.1conv"
        ),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="1.3pool"),
        # Nain's model has SeparableConv2D layers here, but the VGG16 model has Conv2D layers
        # I don't think tensorflow allows transferring regular conv2d weights to separable conv2d layers
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.1conv"
        ),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="2.3pool"),
        # the xception paper says every convolutional layer is followed by a batch normalization layer
        # i'm willing to bet modern techniques might have moved away from this, but we'll try it anyway
        # however, we make an exception for the pretrained layers, since they're already trained
        # the rest of the convolutional layers in Xception are blocks of 2-3 depthwise separable layers, followed by a pooling layer
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.5conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="3.6pool"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.5conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="4.6pool"),
        # all the CNN models end with dense layers, probably to help the model "interpret" what the convolutional layers see
        tf.keras.layers.Flatten(name="5.1flatten"),
        # we almost certainly don't need these many neurons, but nain used that many and maybe he knows something I don't
        tf.keras.layers.Dense(1024, activation="relu", name="5.2dense"),
        # the above dense layer has 118 million of the 119 million trainable parameters!!!
        # add in a dropout layer with a high rate to prevent overfitting
        tf.keras.layers.Dropout(0.7, name="5.3dropout"),
        tf.keras.layers.Dense(512, activation="relu", name="5.4dense"),
        tf.keras.layers.Dropout(0.7, name="5.5dropout"),
        # 1 output neuron with sigmoid is equivalent to 2 output neurons with softmax, but trains faster:
        # https://stats.stackexchange.com/questions/207049/neural-network-for-binary-classification-use-1-or-2-output-neurons
        # tf.keras.layers.Dense(2, activation="softmax", name="output")
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ],
    name="Model_2",
)

# let's use Adam instead of AdamW
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.TruePositives(name="tp"),
        tf.keras.metrics.TrueNegatives(name="tn"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    ],
)

model.get_layer("1.1conv").set_weights(vgg16.get_layer("block1_conv1").get_weights())
model.get_layer("1.2conv").set_weights(vgg16.get_layer("block1_conv2").get_weights())
model.get_layer("2.1conv").set_weights(vgg16.get_layer("block2_conv1").get_weights())
model.get_layer("2.2conv").set_weights(vgg16.get_layer("block2_conv2").get_weights())
model.get_layer("1.1conv").trainable = False
model.get_layer("1.2conv").trainable = False
model.get_layer("2.1conv").trainable = False
model.get_layer("2.2conv").trainable = False

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks(os.path.join(CKPT_ROOT, "best_model_transfer_2.keras")),
)

In [ ]:
summary_graphics(history, model, test_ds)

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} (duration: {notebook_end_time - notebook_start_time})"
)